# Notebook 13 — Local ViT-B/16 full-study inference

**Phase 2E** runs the restored final ViT-B/16 model over every valid slice in one complete local study (up to 36 slices), then exports 12 target probabilities.

Source of truth: final Kaggle inference notebook `06b_vit_residual_attention_submission_kaggle.ipynb`, model call and probability conversion in its inference section:

```python
logits = model(images, valid, plane_ids, fluid, fat, positions)
probs = torch.sigmoid(logits.float())
```

The model itself performs `/255`, grayscale-to-RGB replication, ImageNet normalization, slice chunking, slice encoding and study-level aggregation. This notebook does **not** apply another image normalization and does not use the one-slice smoke-test output from Notebook 12.

This is a one-study software demonstration, not an independent validation estimate or a clinical diagnosis.

In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rsna_knee").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/rsna_knee")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rsna_knee.models.vit_study import TARGETS
from rsna_knee.models.vit_checkpoint import resolve_vit_artifact, load_vit_model
from rsna_knee.inference.vit_full_study import (
    ViTInferenceError,
    predict_vit_full_study,
    save_vit_full_study_outputs,
    validate_vit_inputs,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Runtime-only memory control; it does not alter checkpoint weights or model math.
RUNTIME_SLICE_CHUNK_SIZE = 2 if DEVICE.type == "cpu" else 8

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Runtime slice chunk:", RUNTIME_SLICE_CHUNK_SIZE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Python: 3.13.9
PyTorch: 2.11.0+cpu
Device: cpu
Runtime slice chunk: 2


## 2. Load and validate the Phase 2D preprocessing cache

The cache must contain all six arrays emitted by Notebook 11, plus study/plane/series metadata. The full 36-slot tensor is passed to the model; the validity mask decides which slices are encoded.

In [2]:
PREPROCESSED_PATH = (
    PROJECT_ROOT / "demo" / "output" / "vit_preprocessing"
    / "vit_preprocessed_sample.npz"
)
if not PREPROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"ViT preprocessing cache not found: {PREPROCESSED_PATH}. "
        "Run notebooks/11_local_vit_preprocessing.ipynb first."
    )

with np.load(PREPROCESSED_PATH, allow_pickle=False) as data:
    payload = {
        name: data[name].copy()
        for name in ("images", "valid", "plane_ids", "fluid", "fat", "positions")
    }
    saved_plane_order = data["plane_order"].astype(str).tolist()
    series_uids = data["series_uids"].astype(str).tolist()
    study_uid = str(data["study_uid"].item())

assert saved_plane_order == ["Sagittal", "Coronal", "Axial"], saved_plane_order
payload = validate_vit_inputs(payload)
valid_np = payload["valid"].astype(bool)
plane_ids_np = payload["plane_ids"].astype(np.int64)
valid_by_plane = {
    plane: int(np.sum(valid_np & (plane_ids_np == plane_id)))
    for plane_id, plane in enumerate(("Sagittal", "Coronal", "Axial"))
}

print("Study UID:", study_uid)
print("Images:", payload["images"].shape, payload["images"].dtype)
print("Valid slices:", int(valid_np.sum()), "/ 36")
print("Valid slices by plane:", valid_by_plane)
print("Selected series:", dict(zip(saved_plane_order, series_uids)))
print("[PASS] Phase 2D input contract")

Study UID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Images: (36, 224, 224) uint8
Valid slices: 36 / 36
Valid slices by plane: {'Sagittal': 12, 'Coronal': 12, 'Axial': 12}
Selected series: {'Sagittal': '1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429', 'Coronal': '1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198', 'Axial': '1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430'}
[PASS] Phase 2D input contract


## 3. Resolve the final ViT artifact

Use the final `vit_pretrained_centerprior_model_artifacts.zip` (or its extracted `.pt` checkpoint), not an earlier CLS-only ViT artifact.

In [3]:
configured_artifact = (
    PROJECT_ROOT / "artifacts" / "checkpoints" / "102526"
    / "vit_pretrained_centerprior_model_artifacts.zip"
)
VIT_ARTIFACT_SOURCE = configured_artifact if configured_artifact.exists() else None
vit_artifact = resolve_vit_artifact(VIT_ARTIFACT_SOURCE, PROJECT_ROOT)

print("ViT artifact:", vit_artifact)
print("Artifact size (GiB):", round(vit_artifact.stat().st_size / 1024**3, 3))
assert vit_artifact.is_file()

ViT artifact: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\artifacts\checkpoints\102526\vit_pretrained_centerprior_model_artifacts.zip
Artifact size (GiB): 0.728


## 4. Restore the trained ViT-B/16 checkpoint

The Phase 2D loader instantiates `torchvision.models.vit_b_16(weights=None)` and restores the complete fine-tuned model state with `strict=True`. This notebook reuses that loader; no pretrained weights are downloaded.

In [4]:
restore_started = time.perf_counter()
model, model_info = load_vit_model(
    vit_artifact,
    device=DEVICE,
    runtime_slice_chunk_size=RUNTIME_SLICE_CHUNK_SIZE,
)
restore_seconds = time.perf_counter() - restore_started

print("========== VIT CHECKPOINT RESTORE ==========")
print("Architecture:", model_info["architecture"])
print("Best epoch:", model_info["best_epoch"])
print("Best validation BCE:", model_info["best_val_loss"])
print("Strict state-dict load:", model_info["strict_load"])
print("Parameters (M):", round(model_info["parameter_count"] / 1e6, 3))
print("Device:", model_info["device"])
print("Runtime slice chunk:", model.config.slice_chunk_size)
print("Restore seconds:", round(restore_seconds, 2))
assert model_info["strict_load"] is True
assert model.training is False
print("[PASS] final ViT checkpoint restored with strict=True")

========== VIT CHECKPOINT RESTORE ==========
Architecture: Pretrained ViT-B/16 + gated metadata + study Transformer + residual mean-attention pooling
Best epoch: 7
Best validation BCE: 0.6440770427385966
Strict state-dict load: True
Parameters (M): 88.788
Device: cpu
Runtime slice chunk: 2
Restore seconds: 19.02
[PASS] final ViT checkpoint restored with strict=True


## 5. Run full-study inference

Unlike Notebook 12's one-valid-slice smoke test, this cell passes the original validity mask for **all valid slices**. The model's internal `slice_chunk_size` limits encoder memory while preserving the complete study-level aggregation. The call order and sigmoid conversion match the final Kaggle inference path.

In [5]:
inference_started = time.perf_counter()
prediction = predict_vit_full_study(
    model,
    payload,
    study_uid=study_uid,
    device=DEVICE,
)
inference_seconds = time.perf_counter() - inference_started

print("========== FULL-STUDY VIT INFERENCE ==========")
print("Study UID:", prediction.study_uid)
print("Valid slices processed:", prediction.n_valid_slices, "/ 36")
print("Slices by plane:", prediction.valid_slices_by_plane)
print("Logits shape:", prediction.logits.shape)
print("Probabilities shape:", prediction.probabilities.shape)
print("Probability range:", float(prediction.probabilities.min()), "to", float(prediction.probabilities.max()))
print("Inference seconds:", round(inference_seconds, 2))
assert prediction.n_valid_slices == int(valid_np.sum())
assert prediction.logits.shape == (len(TARGETS),)
assert prediction.probabilities.shape == (len(TARGETS),)
assert np.isfinite(prediction.logits).all()
assert np.isfinite(prediction.probabilities).all()
assert np.all((prediction.probabilities >= 0) & (prediction.probabilities <= 1))
print("[PASS] all valid slices processed; 12 finite probabilities returned")

========== FULL-STUDY VIT INFERENCE ==========
Study UID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Valid slices processed: 36 / 36
Slices by plane: {'Sagittal': 12, 'Coronal': 12, 'Axial': 12}
Logits shape: (12,)
Probabilities shape: (12,)
Probability range: 0.20963922142982483 to 0.4102228283882141
Inference seconds: 5.0
[PASS] all valid slices processed; 12 finite probabilities returned


## 6. Save full-study prediction artifacts

The CSV is a readable one-row-per-target result. The NPZ keeps numeric logits/probabilities for the next ensemble phase. The JSON report records the checkpoint provenance, valid slice counts, timing and exact inference contract.

In [6]:
OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "vit_full_inference"
output_paths = save_vit_full_study_outputs(
    prediction,
    OUTPUT_DIR,
    model_info=model_info,
    restore_seconds=restore_seconds,
    inference_seconds=inference_seconds,
    cache_path=PREPROCESSED_PATH,
)

print("Saved CSV:", output_paths["csv"])
print("Saved NPZ:", output_paths["npz"])
print("Saved report:", output_paths["report"])

display(pd.read_csv(output_paths["csv"]))
report = json.loads(output_paths["report"].read_text(encoding="utf-8"))
assert report["status"] == "success"
assert report["strict_load"] is True
assert report["output"]["all_finite"] is True

Saved CSV: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_full_inference\vit_full_study_predictions.csv
Saved NPZ: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_full_inference\vit_full_study_prediction.npz
Saved report: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_full_inference\vit_full_study_inference_report.json


,target,logit,probability
0,ACL,-0.721200,0.327129
1,MCL,-1.327101,0.209639
2,Medial Meniscus,-0.780842,0.314138
3,Lateral Meniscus,-0.684682,0.335217
4,Medial OA,-0.654749,0.341920
5,Lateral OA,-0.669647,0.338576
6,PF OA,-0.363044,0.410223
7,Effusion,-0.781791,0.313934
8,Synovitis,-0.945855,0.279719
9,Baker's,-0.596694,0.355100


## 7. CRITERIA — Phase 2E full-study ViT inference

All criteria must pass before moving to the 90/10 ensemble phase.

In [7]:
criteria = {
    "Phase 2D cache contract is valid": (
        payload["images"].shape == (36, 224, 224)
        and payload["images"].dtype == np.uint8
        and int(valid_np.sum()) > 0
    ),
    "Final ViT artifact resolved": vit_artifact.is_file(),
    "Checkpoint restored with strict=True": model_info["strict_load"] is True,
    "Full validity mask passed to inference": (
        prediction.n_valid_slices == int(valid_np.sum())
    ),
    "Model returned 12 logits": prediction.logits.shape == (12,),
    "Model returned 12 finite probabilities": (
        prediction.probabilities.shape == (12,)
        and np.isfinite(prediction.probabilities).all()
    ),
    "Probabilities are within [0, 1]": bool(
        np.all((prediction.probabilities >= 0.0) & (prediction.probabilities <= 1.0))
    ),
    "All three output files saved": all(path.is_file() for path in output_paths.values()),
    "Inference report status is successful": report.get("status") == "success",
}

for description, passed in criteria.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {description}")

if not all(criteria.values()):
    raise AssertionError("Phase 2E criteria did not all pass.")

print("\nPHASE 2E: ALL CRITERIA PASSED")

[PASS] Phase 2D cache contract is valid
[PASS] Final ViT artifact resolved
[PASS] Checkpoint restored with strict=True
[PASS] Full validity mask passed to inference
[PASS] Model returned 12 logits
[PASS] Model returned 12 finite probabilities
[PASS] Probabilities are within [0, 1]
[PASS] All three output files saved
[PASS] Inference report status is successful

PHASE 2E: ALL CRITERIA PASSED
